In [1]:
# ==========================================
# FINAL OPTIMIZED CATBOOST TRAINING (REGRESSION)
# ==========================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import time

# Changed to Regressor
from catboost import CatBoostRegressor

# Changed to Regression Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# 1. LOAD DATA
# ==========================================

df_train = pd.read_csv('cargo_routing_train.csv')
df_val = pd.read_csv('cargo_routing_validation.csv')

# ==========================================
# 2. TARGET CREATION (PURE REGRESSION)
# ==========================================

# Threshold removed - predicting the exact score
cols_to_drop = [
    'record_id',
    'timestamp',
    'shipment_id',
    'candidate_route_id',
    'optimization_score'
]

X_train_raw = df_train.drop(columns=cols_to_drop)
y_train = df_train['optimization_score']

X_val_raw = df_val.drop(columns=cols_to_drop)
y_val = df_val['optimization_score']

# ==========================================
# 3. IDENTIFY CATEGORICAL FEATURES
# ==========================================

categorical_features = X_train_raw.select_dtypes(
    include=['object']
).columns.tolist()

print("\n📌 CATEGORICAL FEATURES:")
print(categorical_features)

# ==========================================
# 4. FINAL BEST MODEL
# ==========================================

final_model = CatBoostRegressor(
    iterations=100,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,

    loss_function='RMSE', # Updated for Regression
    eval_metric='RMSE',   # Updated for Regression

    cat_features=categorical_features,

    random_state=42,
    verbose=100
)

# ==========================================
# 5. TRAIN MODEL
# ==========================================

print("\n🚀 TRAINING FINAL MODEL...")
start_time = time.time()

final_model.fit(
    X_train_raw,
    y_train
)

training_time = round(
    (time.time() - start_time) / 60,
    2
)

# ==========================================
# 6. VALIDATION PREDICTIONS
# ==========================================

print("\n📊 GENERATING VALIDATION PREDICTIONS...")

y_pred = final_model.predict(X_val_raw)
# predict_proba removed (not applicable for regression)

# ==========================================
# 7. CALCULATE METRICS
# ==========================================

r2 = r2_score(
    y_val, 
    y_pred
)

mse = mean_squared_error(
    y_val, 
    y_pred
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

# ==========================================
# 8. DISPLAY RESULTS
# ==========================================

print("\n")
print("=" * 60)
print("🏆 FINAL CATBOOST RESULTS (REGRESSION)")
print("=" * 60)

print("\n🔥 FINAL HYPERPARAMETERS:")
print("-" * 40)

print("iterations      :", 100)
print("depth           :", 6)
print("learning_rate   :", 0.05)
print("l2_leaf_reg     :", 3)

print("\n📈 VALIDATION METRICS")
print("-" * 40)

print(f"R-Squared (R²)  : {r2:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"MAE             : {mae:.4f}")
print(f"MSE             : {mse:.4f}")

print(f"\n⏱️ TOTAL TRAINING TIME: {training_time} minutes")

# ==========================================
# 9. FEATURE IMPORTANCE
# ==========================================

feature_importance = pd.DataFrame({
    'Feature': X_train_raw.columns,
    'Importance': final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\n")
print("=" * 60)
print("📌 TOP 15 MOST IMPORTANT FEATURES")
print("=" * 60)

print(feature_importance.head(15))

# ==========================================
# 10. SAVE MODEL
# ==========================================

final_model.save_model(
    'cargo_routing_catboost_model.cbm'
)

print("\n💾 MODEL SAVED SUCCESSFULLY:")
print("cargo_routing_catboost_model.cbm")

# ==========================================
# 11. OPTIONAL CSV EXPORT
# ==========================================

feature_importance.to_csv(
    'feature_importance.csv',
    index=False
)

print("\n💾 FEATURE IMPORTANCE CSV SAVED:")
print("feature_importance.csv")


📌 CATEGORICAL FEATURES:
['origin', 'destination', 'cargo_type', 'priority', 'airline', 'season', 'shc_code']

🚀 TRAINING FINAL MODEL...
0:	learn: 0.2211385	total: 223ms	remaining: 22s
99:	learn: 0.1375946	total: 5.58s	remaining: 0us

📊 GENERATING VALIDATION PREDICTIONS...


🏆 FINAL CATBOOST RESULTS (REGRESSION)

🔥 FINAL HYPERPARAMETERS:
----------------------------------------
iterations      : 100
depth           : 6
learning_rate   : 0.05
l2_leaf_reg     : 3

📈 VALIDATION METRICS
----------------------------------------
R-Squared (R²)  : 0.6133
RMSE            : 0.1401
MAE             : 0.1094
MSE             : 0.0196

⏱️ TOTAL TRAINING TIME: 0.1 minutes


📌 TOP 15 MOST IMPORTANT FEATURES
                     Feature  Importance
8            num_connections   39.726011
12         reliability_score   18.393966
15                  cost_sar   15.405252
13        weather_risk_score    8.769586
11     capacity_available_kg    8.446240
14               load_factor    2.865436
3           